In [ ]:
import time
import numpy as np
import requests
import pandas as pd
from tqdm import tqdm

1. Dati da 14 luglio al 31 dicembre

In [ ]:
# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────
API_KEY  = "5ce1fe8e4e6b7a505a5f0e34a0044e6365b910d01338593ff7d64d8a438b67cd"
BASE_URL = "https://api.openaq.org/v3"
HEADERS  = {"X-API-Key": API_KEY}

DATE_TO  = "2025-12-01T00:00:00Z"   # data fine download

# Pollutanti (nome OpenAQ → notazione nel CSV)
POLL_MAP = {
    "co":   "CO",
    "no2":  "NO2",
    "o3":   "O3",
    "so2":  "SO2",
    "pm10": "PM10",
    "pm25": "PM2.5",
}

# Varianti di nome che OpenAQ può restituire per PM2.5
PM25_ALIASES = {"pm25", "pm2.5"}

LABEL_MAP = {
    "CO":   "Carbon monoxide (air)",
    "NO2":  "Nitrogen dioxide (air)",
    "O3":   "Ozone (air)",
    "SO2":  "Sulphur dioxide (air)",
    "PM10": "Particulate matter < 10 µm (aerosol)",
    "PM2.5":"Particulate matter < 2.5 µm (aerosol)",
}

MATCH_RADIUS_KM = 0.5

INPUT_CSV  = "dati_tesi_incompleti.csv"   # cambia se stai usando dati_tesi_completi.csv
OUTPUT_CSV = "dati_tesi_completi1.csv"

# ─────────────────────────────────────────────
# UTILITIES
# ─────────────────────────────────────────────
def haversine_vec(lat1, lon1, lat2_arr, lon2_arr):
    R = 6371
    dlat = np.radians(lat2_arr - lat1)
    dlon = np.radians(lon2_arr - lon1)
    a = (np.sin(dlat / 2) ** 2
         + np.cos(np.radians(lat1))
         * np.cos(np.radians(lat2_arr))
         * np.sin(dlon / 2) ** 2)
    return R * 2 * np.arcsin(np.sqrt(a))


def api_get(path, params=None, retries=3):
    url = f"{BASE_URL}{path}"
    for attempt in range(retries):
        try:
            r = requests.get(url, headers=HEADERS, params=params, timeout=30)
            if r.status_code == 429:
                wait = int(r.headers.get("x-ratelimit-reset", 60))
                print(f"  Rate limit → attendo {wait}s")
                time.sleep(wait)
                continue
            r.raise_for_status()
            remaining = int(r.headers.get("x-ratelimit-remaining", 60))
            if remaining < 5:
                time.sleep(2)
            return r.json()
        except requests.RequestException as e:
            print(f"  Errore (tentativo {attempt + 1}): {e}")
            time.sleep(5)
    return None


# ─────────────────────────────────────────────
# 1. CARICA IL DATASET
# ─────────────────────────────────────────────
print("Carico dataset originale...")
df_orig = pd.read_csv(INPUT_CSV)
df_orig["data_record_start_time"] = pd.to_datetime(df_orig["data_record_start_time"])
df_orig["data_record_end_time"]   = pd.to_datetime(df_orig["data_record_end_time"])

# DATE_FROM = max(data_record_end_time) nel dataset
date_from_ts = df_orig["data_record_end_time"].max()
DATE_FROM    = date_from_ts.strftime("%Y-%m-%dT%H:%M:%SZ")
print(f"  Scarico dal:  {DATE_FROM}")
print(f"  Scarico fino: {DATE_TO}")

# Tutte le stazioni uniche nel dataset (non solo il periodo denso)
stations = (
    df_orig[["station_eu_code", "station_lat", "station_lon", "station_altitude",
             "station_name", "region_name", "municipality_name", "station_municipality"]]
    .drop_duplicates("station_eu_code")
    .reset_index(drop=True)
)
print(f"  Stazioni uniche nel dataset: {len(stations)}")

# Chiave di deduplicazione
existing_keys = set(
    zip(df_orig["station_eu_code"],
        df_orig["pollutant_notation"],
        df_orig["data_record_start_time"])
)

# ─────────────────────────────────────────────
# 2. SCARICA LOCATION OPENAQ — ITALIA
# ─────────────────────────────────────────────
print("\nScarico location OpenAQ — Italia...")
all_locations = []
page = 1
while True:
    data = api_get("/locations", params={
        "country_id": "IT",
        "limit":      1000,
        "page":       page,
        "monitor":    "true",
    })
    if not data or not data.get("results"):
        break
    all_locations.extend(data["results"])
    found_raw      = str(data["meta"]["found"])
    found_is_exact = ">" not in found_raw
    found          = int(found_raw.replace(">", "").strip())
    print(f"  Pagina {page}: {len(all_locations)} location scaricate")
    if found_is_exact and len(all_locations) >= found:
        break
    page += 1
    time.sleep(0.5)

print(f"  Totale location OpenAQ IT: {len(all_locations)}")

# Costruisci df_oaq includendo PM10 e PM2.5
oaq_rows = []
for loc in all_locations:
    coords = loc.get("coordinates", {})
    lat = coords.get("latitude")
    lon = coords.get("longitude")
    if lat is None or lon is None:
        continue
    for sensor in loc.get("sensors", []):
        param_raw = sensor["parameter"]["name"].lower()
        # Normalizza pm2.5 → pm25
        param = "pm25" if param_raw in PM25_ALIASES else param_raw
        if param in POLL_MAP:
            oaq_rows.append({
                "oaq_location_id": loc["id"],
                "oaq_sensor_id":   sensor["id"],
                "oaq_param":       param,
                "oaq_lat":         lat,
                "oaq_lon":         lon,
                "oaq_name":        loc["name"],
            })

df_oaq = pd.DataFrame(oaq_rows)
print(f"  Sensori utili (tutti i pollutanti): {len(df_oaq)}")
print(df_oaq.groupby("oaq_param")["oaq_sensor_id"].count().to_string())

# ─────────────────────────────────────────────
# 3. MATCH GEOGRAFICO (haversine vettorizzato)
# ─────────────────────────────────────────────
print(f"\nMatch geografico (raggio={MATCH_RADIUS_KM} km)...")

oaq_lats = df_oaq["oaq_lat"].values
oaq_lons = df_oaq["oaq_lon"].values

matches = []
for _, st in stations.iterrows():
    dists = haversine_vec(st["station_lat"], st["station_lon"], oaq_lats, oaq_lons)
    mask  = dists <= MATCH_RADIUS_KM
    for idx in np.where(mask)[0]:
        oq = df_oaq.iloc[idx]
        matches.append({
            "station_eu_code": st["station_eu_code"],
            "oaq_sensor_id":   oq["oaq_sensor_id"],
            "oaq_param":       oq["oaq_param"],
            "poll_notation":   POLL_MAP[oq["oaq_param"]],
            "dist_km":         round(dists[idx], 3),
        })

df_matches = pd.DataFrame(matches)
if len(df_matches) == 0:
    print("  ⚠ Nessun match trovato. Prova ad aumentare MATCH_RADIUS_KM.")
    exit()

# Sensore più vicino per ogni (station_eu_code, pollutante)
df_matches = (
    df_matches.sort_values("dist_km")
              .drop_duplicates(["station_eu_code", "poll_notation"])
              .reset_index(drop=True)
)
print(f"  Match trovati: {len(df_matches)}")
print(df_matches.groupby("poll_notation")["station_eu_code"].count().to_string())

# ─────────────────────────────────────────────
# 4. SCARICA MISURAZIONI ORARIE PER OGNI MATCH
# ─────────────────────────────────────────────
print(f"\nScarico misurazioni orarie ({DATE_FROM[:10]} → {DATE_TO[:10]})...")

new_rows = []

for _, m in tqdm(df_matches.iterrows(), total=len(df_matches), desc="Sensori"):
    sensor_id  = int(m["oaq_sensor_id"])
    eu_code    = m["station_eu_code"]
    poll_notat = m["poll_notation"]

    st_meta = stations[stations["station_eu_code"] == eu_code].iloc[0]

    # Unità e label dal CSV originale, altrimenti default
    poll_rows = df_orig[df_orig["pollutant_notation"] == poll_notat]
    if len(poll_rows) > 0:
        poll_label = poll_rows["pollutant_label"].iloc[0]
        unit       = poll_rows["observation_unit_notation"].iloc[0]
    else:
        poll_label = LABEL_MAP.get(poll_notat, poll_notat)
        unit       = "µg/m3"

    page = 1
    while True:
        data = api_get(
            f"/sensors/{sensor_id}/hours",
            params={
                "datetime_from": DATE_FROM,
                "datetime_to":   DATE_TO,
                "limit":         1000,
                "page":          page,
            }
        )
        if not data or not data.get("results"):
            break

        for meas in data["results"]:
            dt_utc = meas.get("period", {}).get("datetimeFrom", {}).get("utc")
            value  = meas.get("value")
            if dt_utc is None or value is None:
                continue

            dt_start = pd.Timestamp(dt_utc).tz_localize(None)
            dt_end   = dt_start + pd.Timedelta(hours=1)

            key = (eu_code, poll_notat, dt_start)
            if key in existing_keys:
                continue

            new_rows.append({
                "region_name":                  st_meta["region_name"],
                "municipality_name":            st_meta["municipality_name"],
                "station_municipality":         st_meta["station_municipality"],
                "station_name":                 st_meta["station_name"],
                "station_eu_code":              eu_code,
                "station_position":             None,
                "station_lat":                  st_meta["station_lat"],
                "station_lon":                  st_meta["station_lon"],
                "station_altitude":             st_meta["station_altitude"],
                "primary_observation_notation": "0 days 01:00:00",
                "data_record_start_time":       dt_start,
                "data_record_end_time":         dt_end,
                "data_record_value":            value,
                "observation_unit_notation":    unit,
                "pollutant_notation":           poll_notat,
                "pollutant_label":              poll_label,
            })

        # Paginazione
        meta      = data.get("meta", {})
        found_raw = str(meta.get("found", 0))
        found     = int(found_raw.replace(">", "").strip())
        if len(data["results"]) < 1000 or page * 1000 >= found:
            break
        page += 1
        time.sleep(0.2)

    time.sleep(0.2)

# ─────────────────────────────────────────────
# 5. MERGE E SALVATAGGIO
# ─────────────────────────────────────────────
print(f"\nNuovi record scaricati da OpenAQ: {len(new_rows):,}")

if len(new_rows) == 0:
    print("Nessun dato nuovo — controlla i match e il range di date.")
else:
    df_new      = pd.DataFrame(new_rows)
    df_enriched = pd.concat([df_orig, df_new], ignore_index=True)

    df_enriched["data_record_start_time"] = pd.to_datetime(df_enriched["data_record_start_time"])
    df_enriched = df_enriched.sort_values(
        ["station_eu_code", "pollutant_notation", "data_record_start_time"]
    ).reset_index(drop=True)

    df_enriched.to_csv(OUTPUT_CSV, index=False)
    print(f"\n✓ Salvato: {OUTPUT_CSV}  ({len(df_enriched):,} record totali)")

    print("\nCopertura per pollutante nel dataset arricchito:")
    for poll, grp in df_enriched.groupby("pollutant_notation"):
        n_orig = len(df_orig[df_orig["pollutant_notation"] == poll])
        n_new  = len(grp) - n_orig
        print(f"  {poll:6s}  originali={n_orig:6,}  aggiunti={n_new:6,}  tot={len(grp):6,}")

2. Dati dal 1 gennaio al 14 giugno

In [ ]:
# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────
API_KEY  = "5ce1fe8e4e6b7a505a5f0e34a0044e6365b910d01338593ff7d64d8a438b67cd"
BASE_URL = "https://api.openaq.org/v3"
HEADERS  = {"X-API-Key": API_KEY}

# ⬇ Imposta qui la data di inizio download (inclusa)
DATE_FROM = "2025-01-01T00:00:00Z"

# DATE_TO viene calcolata automaticamente = min(data_record_start_time) del CSV

POLL_MAP = {
    "co":   "CO",
    "no2":  "NO2",
    "o3":   "O3",
    "so2":  "SO2",
    "pm10": "PM10",
    "pm25": "PM2.5",
}

PM25_ALIASES = {"pm25", "pm2.5"}

LABEL_MAP = {
    "CO":    "Carbon monoxide (air)",
    "NO2":   "Nitrogen dioxide (air)",
    "O3":    "Ozone (air)",
    "SO2":   "Sulphur dioxide (air)",
    "PM10":  "Particulate matter < 10 µm (aerosol)",
    "PM2.5": "Particulate matter < 2.5 µm (aerosol)",
}

MATCH_RADIUS_KM = 0.5

INPUT_CSV  = "dati_tesi_completi1.csv"
OUTPUT_CSV = "dati_tesi_completi6.csv"

# ─────────────────────────────────────────────
# UTILITIES
# ─────────────────────────────────────────────
def haversine_vec(lat1, lon1, lat2_arr, lon2_arr):
    R = 6371
    dlat = np.radians(lat2_arr - lat1)
    dlon = np.radians(lon2_arr - lon1)
    a = (np.sin(dlat / 2) ** 2
         + np.cos(np.radians(lat1))
         * np.cos(np.radians(lat2_arr))
         * np.sin(dlon / 2) ** 2)
    return R * 2 * np.arcsin(np.sqrt(a))


def api_get(path, params=None, retries=3):
    url = f"{BASE_URL}{path}"
    for attempt in range(retries):
        try:
            r = requests.get(url, headers=HEADERS, params=params, timeout=30)
            if r.status_code == 429:
                wait = int(r.headers.get("x-ratelimit-reset", 60))
                print(f"  Rate limit → attendo {wait}s")
                time.sleep(wait)
                continue
            r.raise_for_status()
            remaining = int(r.headers.get("x-ratelimit-remaining", 60))
            if remaining < 5:
                time.sleep(2)
            return r.json()
        except requests.RequestException as e:
            print(f"  Errore (tentativo {attempt + 1}): {e}")
            time.sleep(5)
    return None


# ─────────────────────────────────────────────
# 1. CARICA IL DATASET
# ─────────────────────────────────────────────
print("Carico dataset originale...")
df_orig = pd.read_csv(INPUT_CSV)
df_orig["data_record_start_time"] = pd.to_datetime(df_orig["data_record_start_time"])
df_orig["data_record_end_time"]   = pd.to_datetime(df_orig["data_record_end_time"])

# DATE_TO = min(data_record_start_time) nel dataset
date_to_ts = df_orig["data_record_start_time"].min()
DATE_TO    = date_to_ts.strftime("%Y-%m-%dT%H:%M:%SZ")
print(f"  Scarico dal:  {DATE_FROM}")
print(f"  Scarico fino: {DATE_TO}  ← min(data_record_start_time) del CSV")

if DATE_FROM >= DATE_TO:
    print("\n⚠ DATE_FROM è uguale o successiva a min(data_record_start_time). Nulla da scaricare.")
    exit()

# Tutte le stazioni uniche nel dataset
stations = (
    df_orig[["station_eu_code", "station_lat", "station_lon", "station_altitude",
             "station_name", "region_name", "municipality_name", "station_municipality"]]
    .drop_duplicates("station_eu_code")
    .reset_index(drop=True)
)
print(f"  Stazioni uniche nel dataset: {len(stations)}")

# Chiave di deduplicazione (evita duplicati se si ri-esegue)
existing_keys = set(
    zip(df_orig["station_eu_code"],
        df_orig["pollutant_notation"],
        df_orig["data_record_start_time"])
)

# ─────────────────────────────────────────────
# 2. SCARICA LOCATION OPENAQ — ITALIA
# ─────────────────────────────────────────────
print("\nScarico location OpenAQ — Italia...")
all_locations = []
page = 1
while True:
    data = api_get("/locations", params={
        "country_id": "IT",
        "limit":      1000,
        "page":       page,
        "monitor":    "true",
    })
    if not data or not data.get("results"):
        break
    all_locations.extend(data["results"])
    found_raw      = str(data["meta"]["found"])
    found_is_exact = ">" not in found_raw
    found          = int(found_raw.replace(">", "").strip())
    print(f"  Pagina {page}: {len(all_locations)} location scaricate")
    if found_is_exact and len(all_locations) >= found:
        break
    page += 1
    time.sleep(0.5)

print(f"  Totale location OpenAQ IT: {len(all_locations)}")

# Costruisci df_oaq
oaq_rows = []
for loc in all_locations:
    coords = loc.get("coordinates", {})
    lat = coords.get("latitude")
    lon = coords.get("longitude")
    if lat is None or lon is None:
        continue
    for sensor in loc.get("sensors", []):
        param_raw = sensor["parameter"]["name"].lower()
        param = "pm25" if param_raw in PM25_ALIASES else param_raw
        if param in POLL_MAP:
            oaq_rows.append({
                "oaq_location_id": loc["id"],
                "oaq_sensor_id":   sensor["id"],
                "oaq_param":       param,
                "oaq_lat":         lat,
                "oaq_lon":         lon,
                "oaq_name":        loc["name"],
            })

df_oaq = pd.DataFrame(oaq_rows)
print(f"  Sensori utili (tutti i pollutanti): {len(df_oaq)}")
print(df_oaq.groupby("oaq_param")["oaq_sensor_id"].count().to_string())

# ─────────────────────────────────────────────
# 3. MATCH GEOGRAFICO (haversine vettorizzato)
# ─────────────────────────────────────────────
print(f"\nMatch geografico (raggio={MATCH_RADIUS_KM} km)...")

oaq_lats = df_oaq["oaq_lat"].values
oaq_lons = df_oaq["oaq_lon"].values

matches = []
for _, st in stations.iterrows():
    dists = haversine_vec(st["station_lat"], st["station_lon"], oaq_lats, oaq_lons)
    mask  = dists <= MATCH_RADIUS_KM
    for idx in np.where(mask)[0]:
        oq = df_oaq.iloc[idx]
        matches.append({
            "station_eu_code": st["station_eu_code"],
            "oaq_sensor_id":   oq["oaq_sensor_id"],
            "oaq_param":       oq["oaq_param"],
            "poll_notation":   POLL_MAP[oq["oaq_param"]],
            "dist_km":         round(dists[idx], 3),
        })

df_matches = pd.DataFrame(matches)
if len(df_matches) == 0:
    print("  ⚠ Nessun match trovato. Prova ad aumentare MATCH_RADIUS_KM.")
    exit()

# Sensore più vicino per ogni (station_eu_code, pollutante)
df_matches = (
    df_matches.sort_values("dist_km")
              .drop_duplicates(["station_eu_code", "poll_notation"])
              .reset_index(drop=True)
)
print(f"  Match trovati: {len(df_matches)}")
print(df_matches.groupby("poll_notation")["station_eu_code"].count().to_string())

# ─────────────────────────────────────────────
# 4. SCARICA MISURAZIONI ORARIE PER OGNI MATCH
# ─────────────────────────────────────────────
print(f"\nScarico misurazioni orarie ({DATE_FROM[:10]} → {DATE_TO[:10]})...")

new_rows = []

for _, m in tqdm(df_matches.iterrows(), total=len(df_matches), desc="Sensori"):
    sensor_id  = int(m["oaq_sensor_id"])
    eu_code    = m["station_eu_code"]
    poll_notat = m["poll_notation"]

    st_meta = stations[stations["station_eu_code"] == eu_code].iloc[0]

    # Unità e label dal CSV originale, altrimenti default
    poll_rows = df_orig[df_orig["pollutant_notation"] == poll_notat]
    if len(poll_rows) > 0:
        poll_label = poll_rows["pollutant_label"].iloc[0]
        unit       = poll_rows["observation_unit_notation"].iloc[0]
    else:
        poll_label = LABEL_MAP.get(poll_notat, poll_notat)
        unit       = "µg/m3"

    page = 1
    while True:
        data = api_get(
            f"/sensors/{sensor_id}/hours",
            params={
                "datetime_from": DATE_FROM,
                "datetime_to":   DATE_TO,
                "limit":         1000,
                "page":          page,
            }
        )
        if not data or not data.get("results"):
            break

        for meas in data["results"]:
            dt_utc = meas.get("period", {}).get("datetimeFrom", {}).get("utc")
            value  = meas.get("value")
            if dt_utc is None or value is None:
                continue

            dt_start = pd.Timestamp(dt_utc).tz_localize(None)
            dt_end   = dt_start + pd.Timedelta(hours=1)

            key = (eu_code, poll_notat, dt_start)
            if key in existing_keys:
                continue

            new_rows.append({
                "region_name":                  st_meta["region_name"],
                "municipality_name":            st_meta["municipality_name"],
                "station_municipality":         st_meta["station_municipality"],
                "station_name":                 st_meta["station_name"],
                "station_eu_code":              eu_code,
                "station_position":             None,
                "station_lat":                  st_meta["station_lat"],
                "station_lon":                  st_meta["station_lon"],
                "station_altitude":             st_meta["station_altitude"],
                "primary_observation_notation": "0 days 01:00:00",
                "data_record_start_time":       dt_start,
                "data_record_end_time":         dt_end,
                "data_record_value":            value,
                "observation_unit_notation":    unit,
                "pollutant_notation":           poll_notat,
                "pollutant_label":              poll_label,
            })

        # Paginazione
        meta      = data.get("meta", {})
        found_raw = str(meta.get("found", 0))
        found     = int(found_raw.replace(">", "").strip())
        if len(data["results"]) < 1000 or page * 1000 >= found:
            break
        page += 1
        time.sleep(0.2)

    time.sleep(0.2)

# ─────────────────────────────────────────────
# 5. MERGE E SALVATAGGIO
# ─────────────────────────────────────────────
print(f"\nNuovi record scaricati da OpenAQ: {len(new_rows):,}")

if len(new_rows) == 0:
    print("Nessun dato nuovo — controlla i match e il range di date.")
else:
    df_new      = pd.DataFrame(new_rows)
    df_enriched = pd.concat([df_orig, df_new], ignore_index=True)

    df_enriched["data_record_start_time"] = pd.to_datetime(df_enriched["data_record_start_time"])
    df_enriched = df_enriched.sort_values(
        ["station_eu_code", "pollutant_notation", "data_record_start_time"]
    ).reset_index(drop=True)

    df_enriched.to_csv(OUTPUT_CSV, index=False)
    print(f"\n✓ Salvato: {OUTPUT_CSV}  ({len(df_enriched):,} record totali)")

    print("\nCopertura per pollutante nel dataset arricchito:")
    for poll, grp in df_enriched.groupby("pollutant_notation"):
        n_orig = len(df_orig[df_orig["pollutant_notation"] == poll])
        n_new  = len(grp) - n_orig
        print(f"  {poll:6s}  originali={n_orig:6,}  aggiunti={n_new:6,}  tot={len(grp):6,}")